# Section 7 Foundation, Rescaled to ~1M Parameters (matching Cameron's scale)

**Why this exists.** Cameron's diffusion experiments run at P≈1.01M for a direct, fair comparison
across methods and against his own results. This notebook rebuilds the SAME validated architecture
and pipeline (T=1000 schedule, x0-clipped sampler -- both confirmed bug fixes carried forward
unchanged) at `base_ch=56`, which gives P=1,059,897 -- matching Cameron's scale closely enough for
a direct comparison.

**Nothing about the validated pipeline changes except size.** The noise schedule and sampler are
exactly the fixes already confirmed to work; only `base_ch` increases from 32 to 56.

## Step 0 — Setup

In [ ]:
import json
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)
if device == 'cpu':
    print('WARNING: use a GPU runtime -- this model and training budget are meaningfully larger.')

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

RESULTS_LOG = []

def log_result(name, config, result, seed):
    entry = {'name': name, 'seed': seed, 'config': config, 'result': result,
              'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')}
    RESULTS_LOG.append(entry)
    print(f"[logged] {name}: {result}")
    return entry

def save_provenance(path='provenance_diffusion_1M_foundation.json'):
    with open(path, 'w') as f:
        json.dump(RESULTS_LOG, f, indent=2, default=str)
    print(f'Saved {len(RESULTS_LOG)} logged results to {path}')


## Step 1 — Noise schedule (T=1000, the confirmed fix)

In [ ]:
T_STEPS = 1000
beta_start, beta_end = 1e-4, 0.02
betas = torch.linspace(beta_start, beta_end, T_STEPS, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

def forward_diffusion(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    sqrt_ab = alpha_bars[t].sqrt().view(-1, 1, 1, 1)
    sqrt_1mab = (1 - alpha_bars[t]).sqrt().view(-1, 1, 1, 1)
    return sqrt_ab * x0 + sqrt_1mab * noise, noise

assert alpha_bars[-1].item() < 0.01, 'Schedule does not reach near-pure noise.'
print(f'T={T_STEPS}, alpha_bar[-1]={alpha_bars[-1].item():.6f} -- OK')


## Step 2 — The denoiser, rescaled to ~1M parameters (base_ch=56)

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-np.log(10000) * torch.arange(half, device=t.device).float() / half)
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(min(8, out_ch), out_ch)
        self.norm2 = nn.GroupNorm(min(8, out_ch), out_ch)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.act = nn.SiLU()
    def forward(self, x, t_emb):
        h = self.act(self.norm1(self.conv1(x)))
        h = h + self.time_proj(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = self.act(self.norm2(self.conv2(h)))
        return h

class SmallUNet(nn.Module):
    def __init__(self, base_ch=56, time_dim=32):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim), nn.SiLU(), nn.Linear(time_dim, time_dim)
        )
        self.enc1 = ConvBlock(1, base_ch, time_dim)
        self.down1 = nn.Conv2d(base_ch, base_ch, 3, stride=2, padding=1)
        self.enc2 = ConvBlock(base_ch, base_ch * 2, time_dim)
        self.down2 = nn.Conv2d(base_ch * 2, base_ch * 2, 3, stride=2, padding=1)
        self.bottleneck = ConvBlock(base_ch * 2, base_ch * 2, time_dim)
        self.up2 = nn.ConvTranspose2d(base_ch * 2, base_ch * 2, 4, stride=2, padding=1)
        self.dec2 = ConvBlock(base_ch * 4, base_ch, time_dim)
        self.up1 = nn.ConvTranspose2d(base_ch, base_ch, 4, stride=2, padding=1)
        self.dec1 = ConvBlock(base_ch * 2, base_ch, time_dim)
        self.out_conv = nn.Conv2d(base_ch, 1, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_embed(t)
        e1 = self.enc1(x, t_emb)
        e2 = self.enc2(self.down1(e1), t_emb)
        b = self.bottleneck(self.down2(e2), t_emb)
        d2 = self.up2(b)
        d2 = self.dec2(torch.cat([d2, e2], dim=1), t_emb)
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1), t_emb)
        return self.out_conv(d1)

denoiser = SmallUNet(base_ch=56, time_dim=32).to(device)
P_denoiser = sum(p.numel() for p in denoiser.parameters())
print(f'Denoiser parameter count: P = {P_denoiser:,}  (target: ~1.01M, matching Cameron\'s scale)')
log_result('denoiser_1M_build', {'base_ch': 56, 'time_dim': 32}, {'param_count': P_denoiser}, SEED)


## Step 3 — Real MNIST data (train + held-out test split)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x * 2 - 1),
])
mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

N_TRAIN = 8000
train_idx = torch.randperm(len(mnist_train))[:N_TRAIN]
X_train = torch.stack([mnist_train[i][0] for i in train_idx]).to(device)

N_TEST = 500
test_idx = torch.randperm(len(mnist_test))[:N_TEST]
X_test = torch.stack([mnist_test[i][0] for i in test_idx]).to(device)

print(f'Training images: {X_train.shape}, held-out test images: {X_test.shape}')


## Step 4 — The sampler (x0-clipping, the confirmed fix)

In [ ]:
@torch.no_grad()
def sample_ddpm(model, n_samples=8, seed=None):
    model.eval()
    if seed is not None:
        gen = torch.Generator(device=device).manual_seed(seed)
        x = torch.randn(n_samples, 1, 28, 28, device=device, generator=gen)
    else:
        x = torch.randn(n_samples, 1, 28, 28, device=device)
    for t_step in reversed(range(T_STEPS)):
        t_batch = torch.full((n_samples,), t_step, device=device, dtype=torch.long)
        pred_noise = model(x, t_batch)
        alpha_t = alphas[t_step]; alpha_bar_t = alpha_bars[t_step]
        alpha_bar_prev = alpha_bars[t_step - 1] if t_step > 0 else torch.tensor(1.0, device=device)
        beta_t = betas[t_step]
        x0_pred = ((x - (1 - alpha_bar_t).sqrt() * pred_noise) / alpha_bar_t.sqrt()).clamp(-1.0, 1.0)
        posterior_mean = (
            (alpha_bar_prev.sqrt() * beta_t / (1 - alpha_bar_t)) * x0_pred
            + (alpha_t.sqrt() * (1 - alpha_bar_prev) / (1 - alpha_bar_t)) * x
        )
        posterior_var = beta_t * (1 - alpha_bar_prev) / (1 - alpha_bar_t)
        x = posterior_mean + posterior_var.sqrt() * torch.randn_like(x) if t_step > 0 else posterior_mean
    model.train()
    return x

def show_sample_grid(samples, title, save_path):
    disp = (samples.clamp(-1, 1) + 1) / 2
    fig, axes = plt.subplots(1, 8, figsize=(16, 2))
    for i, ax in enumerate(axes):
        ax.imshow(disp[i, 0].cpu().numpy(), cmap='gray'); ax.axis('off')
    plt.suptitle(title); plt.savefig(save_path, dpi=120, bbox_inches='tight'); plt.show()


## Step 5 — Training: warmup + cosine LR schedule, checkpointed every 3,000 steps

Same recipe that worked at base_ch=32 (18,000 steps, warmup+cosine decay). Given the larger model,
allow the same step count first and check quality before deciding whether more is needed.

In [ ]:
import math

BATCH_SIZE = 128
N_STEPS = 18000
PEAK_LR = 2e-4
WARMUP_STEPS = 500
SAMPLE_EVERY = 3000
CHECKPOINT_SEED = 12345

opt = torch.optim.Adam(denoiser.parameters(), lr=PEAK_LR)

def lr_at(step):
    if step < WARMUP_STEPS:
        return PEAK_LR * step / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, (N_STEPS - WARMUP_STEPS))
    return PEAK_LR * 0.5 * (1 + math.cos(math.pi * progress))

losses = []
t0 = time.time()
for step in range(N_STEPS):
    lr_now = lr_at(step)
    for g in opt.param_groups:
        g['lr'] = lr_now
    idx = torch.randint(0, X_train.shape[0], (BATCH_SIZE,), device=device)
    x0 = X_train[idx]
    t = torch.randint(0, T_STEPS, (BATCH_SIZE,), device=device)
    x_t, noise = forward_diffusion(x0, t)
    opt.zero_grad(set_to_none=True)
    pred_noise = denoiser(x_t, t)
    loss = F.mse_loss(pred_noise, noise)
    loss.backward()
    opt.step()
    losses.append(loss.item())

    if (step + 1) % 500 == 0:
        print(f'step {step+1}/{N_STEPS}  loss={np.mean(losses[-500:]):.4f}  lr={lr_now:.2e}')

    if (step + 1) % SAMPLE_EVERY == 0:
        print(f'  ...checkpoint at step {step+1}: sample grid + saving weights...')
        ckpt_samples = sample_ddpm(denoiser, n_samples=8, seed=CHECKPOINT_SEED)
        show_sample_grid(ckpt_samples, f'Checkpoint at step {step+1}/{N_STEPS}',
                          f'denoiser_1M_checkpoint_step{step+1}.png')
        ckpt_path = f'denoiser_1M_checkpoint_step{step+1}.pt'
        torch.save({'model_state_dict': denoiser.state_dict(), 'step': step + 1,
                    'loss_last_500': float(np.mean(losses[-500:]))}, ckpt_path)
        log_result('denoiser_1M_checkpoint', {'step': step + 1},
                   {'loss_last_500': float(np.mean(losses[-500:])), 'checkpoint_path': ckpt_path}, SEED)

elapsed = time.time() - t0
print(f'\nDone in {elapsed:.1f}s. Final loss (last 100 steps): {np.mean(losses[-100:]):.4f}')
log_result('denoiser_1M_training', {'n_steps': N_STEPS, 'peak_lr': PEAK_LR, 'base_ch': 56, 'param_count': P_denoiser},
           {'final_loss': float(np.mean(losses[-100:])), 'wall_clock_s': elapsed}, SEED)


## Step 6 — Pick the best checkpoint, and run quantitative reconstruction metrics

**Pixel-accuracy, PSNR, SSIM** (Cameron's metrics), computed as a proper RECONSTRUCTION benchmark:
take real held-out test images, add a fixed amount of noise at a chosen timestep, and check how well
the model's single-shot $x_0$ estimate recovers the original -- this is what makes these metrics
well-defined (full unconditional generation from pure noise has no paired ground truth to compare
against pixel-wise).

In [ ]:
BEST_STEP = N_STEPS   # <-- change to an earlier checkpoint if it looked better

ckpt = torch.load(f'denoiser_1M_checkpoint_step{BEST_STEP}.pt', map_location=device)
denoiser.load_state_dict(ckpt['model_state_dict'])
print(f"Reloaded weights from step {ckpt['step']} (loss at save: {ckpt['loss_last_500']:.4f})")

def ssim_simple(img1, img2, C1=0.01**2, C2=0.03**2):
    # Simplified global SSIM (not windowed) -- adequate for a quick benchmark, not a substitute
    # for a proper windowed SSIM implementation if this needs to go in a final paper table.
    mu1, mu2 = img1.mean(), img2.mean()
    var1, var2 = img1.var(), img2.var()
    covar = ((img1 - mu1) * (img2 - mu2)).mean()
    return ((2*mu1*mu2 + C1) * (2*covar + C2)) / ((mu1**2 + mu2**2 + C1) * (var1 + var2 + C2))

@torch.no_grad()
def reconstruction_metrics(model, X, t_val, n_samples=100):
    model.eval()
    x0 = X[:n_samples]
    t = torch.full((n_samples,), t_val, device=device)
    x_t, noise = forward_diffusion(x0, t)
    pred_noise = model(x_t, t)
    alpha_bar_t = alpha_bars[t_val]
    x0_pred = ((x_t - (1 - alpha_bar_t).sqrt() * pred_noise) / alpha_bar_t.sqrt()).clamp(-1, 1)

    x0_disp = (x0.clamp(-1, 1) + 1) / 2
    x0_pred_disp = (x0_pred.clamp(-1, 1) + 1) / 2

    mse = F.mse_loss(x0_pred_disp, x0_disp).item()
    psnr = 10 * np.log10(1.0 / max(mse, 1e-10))
    pixel_acc = ((x0_pred_disp > 0.5) == (x0_disp > 0.5)).float().mean().item()
    ssim_val = ssim_simple(x0_pred_disp, x0_disp).item()
    model.train()
    return {'mse': mse, 'psnr_db': psnr, 'pixel_accuracy': pixel_acc, 'ssim': ssim_val}

for t_val in [100, 300, 500, 700, 900]:
    m = reconstruction_metrics(denoiser, X_test, t_val)
    print(f't={t_val:>4}: pixel-acc={m["pixel_accuracy"]:.4f}  PSNR={m["psnr_db"]:.2f}dB  SSIM={m["ssim"]:.4f}')
    log_result('denoiser_1M_reconstruction_metrics', {'t': t_val, 'best_step': BEST_STEP}, m, SEED)


## Step 7 — Final generation check (fixed seed + fresh noise)

In [ ]:
samples_fixed = sample_ddpm(denoiser, n_samples=8, seed=CHECKPOINT_SEED)
show_sample_grid(samples_fixed, 'FINAL (1M): same seed as training checkpoints', 'denoiser_1M_final_fixedseed.png')

samples_random = sample_ddpm(denoiser, n_samples=8)
show_sample_grid(samples_random, 'FINAL (1M): fresh random noise', 'denoiser_1M_final_random.png')

log_result('denoiser_1M_final_samples', {'best_step': BEST_STEP}, {'generated': True}, SEED)


## Step 8 — Save provenance

In [ ]:
save_provenance('provenance_diffusion_1M_foundation.json')
print()
for entry in RESULTS_LOG:
    print(f"  - {entry['name']}: {entry['result']}")


## Summary

This gives you a ~1.01M-parameter validated denoiser (matching Cameron's scale) with real
quantitative reconstruction metrics (pixel-accuracy, PSNR, SSIM) alongside the generative check.
Pick the best checkpoint by eye (same principle as before -- quality is not guaranteed monotonic in
loss), then use this checkpoint as the foundation for the rebuilt Test 1 notebook, which adds
Cameron's suggested low-M/many-step ("noisy") budget allocation alongside the original.